#Project Title:
**Enterprise Talent Acquisition & Recruitment Efficiency Dashboard**

#Buisness Problem
A rapidly growing tech company is struggling with a high Time-to-Hire and skyrocketing Cost-per-Hire. The VP of Talent Acquisition needs a data-driven solution to identify bottlenecks in the hiring pipeline, evaluate the ROI of various recruitment channels (LinkedIn, agencies, job boards, etc), and track Diversity, Equity, and Inclusion (DEI) targets.

**Phase 1:** **Data Architecture & Datasets Generation.**

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
num_applications = 2500

# 1. Generate Dim_Recruiters
recruiters_data = {
    "RecruiterID": [f"REC{i:03d}" for i in range(1, 9)],
    "RecruiterName": ["Alex Wong", "Sarah Jenkins", "Michael Chang", "Emily Rodriguez", "David Kim", "Jessica Taylor", "James Wilson", "Amanda Martinez"],
    "Region": ["AMER", "AMER", "APAC", "EMEA", "APAC", "EMEA", "AMER", "EMEA"],
    "Team": ["Tech", "Non-Tech", "Tech", "Tech", "Non-Tech", "Tech", "Executive", "Non-Tech"]
}
dim_recruiters = pd.DataFrame(recruiters_data)

# 2. Generate Dim_Jobs
departments = ["Engineering", "Product", "Data & Analytics", "Sales", "Marketing", "HR & Ops"]
roles = {
    "Engineering": ["Software Engineer", "Senior Frontend Developer", "DevOps Engineer", "Engineering Manager"],
    "Product": ["Product Manager", "Senior PM"],
    "Data & Analytics": ["Data Analyst", "Data Scientist", "Analytics Engineer"],
    "Sales": ["Account Executive", "Business Development Rep"],
    "Marketing": ["Growth Marketer", "SEO Specialist"],
    "HR & Ops": ["HR Specialist", "Operations Coordinator"]
}

jobs_list = []
for i in range(1, 46):
    dept = np.random.choice(departments)
    role = np.random.choice(roles[dept])
    salary = np.random.randint(60, 140) * 1000 if "Senior" not in role else np.random.randint(130, 200) * 1000
    if "Manager" in role: salary = np.random.randint(160, 240) * 1000

    open_date = datetime(2025, 1, 1) + timedelta(days=int(np.random.randint(0, 450)))  # Open dates spread across 2025 and early 2026
    jobs_list.append({
        "JobID": f"JOB{i:03d}",
        "Department": dept,
        "RoleTitle": role,
        "TargetSalary": salary,
        "DateOpened": open_date.strftime("%Y-%m-%d"),
        "RecruiterID": np.random.choice(dim_recruiters["RecruiterID"])
    })
dim_jobs = pd.DataFrame(jobs_list)

# 3. Generate Fact_Applications
sources = ["LinkedIn", "Indeed", "Company Website", "Employee Referral", "Recruitment Agency"]
source_weights = [0.45, 0.25, 0.15, 0.10, 0.05]
stages = ["Applied", "Screening", "Technical Test", "Hiring Manager Interview", "Offer Extended", "Hired", "Rejected", "Withdrawn"]

apps_list = []
interview_logs = []
log_id_counter = 1

for i in range(1, num_applications + 1):
    job = dim_jobs.sample(1).iloc[0]
    source = np.random.choice(sources, p=source_weights)
    app_date = datetime.strptime(job["DateOpened"], "%Y-%m-%d") + timedelta(days=int(np.random.randint(1, 45)))

    # Determine candidate progression
    rand_val = np.random.rand()
    if job["Department"] == "Engineering" and rand_val < 0.50:
        final_stage = "Technical Test"
        status = "Rejected"
    elif rand_val < 0.12:
        final_stage = "Hired"
        status = "Hired"
    elif rand_val < 0.15:
        final_stage = "Offer Extended"
        status = "Withdrawn"  # Declined offer
    elif rand_val < 0.65:
        final_stage = np.random.choice(["Screening", "Technical Test", "Hiring Manager Interview"])
        status = "Rejected"
    else:
        final_stage = np.random.choice(["Screening", "Technical Test", "Hiring Manager Interview"])
        status = "Withdrawn"

    # logic to build operational logs
    stage_order = ["Applied", "Screening", "Technical Test", "Hiring Manager Interview", "Offer Extended", "Hired"]
    current_date = app_date

    for stage in stage_order:
        # Log this interview step
        interview_logs.append({
            "LogID": f"LOG{log_id_counter:06d}",
            "ApplicationID": f"APP{i:05d}",
            "Stage": stage,
            "StepDate": current_date.strftime("%Y-%m-%d"),
            "InterviewerRating": np.random.randint(1, 6) if stage in ["Technical Test", "Hiring Manager Interview"] else None
        })
        log_id_counter += 1

        if stage == final_stage:
            if status in ["Rejected", "Withdrawn"]:
                # Add the final exit log
                interview_logs.append({
                    "LogID": f"LOG{log_id_counter:06d}",
                    "ApplicationID": f"APP{i:05d}",
                    "Stage": status,
                    "StepDate": (current_date + timedelta(days=np.random.randint(1, 5))).strftime("%Y-%m-%d"),
                    "InterviewerRating": None
                })
                log_id_counter += 1
            break

        # Advance time to next stage
        current_date += timedelta(days=int(np.random.randint(2, 12)))

    # Track financial agency costs
    agency_cost = 0
    if source == "Recruitment Agency" and status == "Hired":
        agency_cost = int(job["TargetSalary"] * 0.20) # 20% agency fee
    elif source == "LinkedIn":
        agency_cost = 15 # cost per application click

    apps_list.append({
        "ApplicationID": f"APP{i:05d}",
        "CandidateID": f"CAND{np.random.randint(10000, 99999)}",
        "JobID": job["JobID"],
        "SourcingChannel": source,
        "ApplicationDate": app_date.strftime("%Y-%m-%d"),
        "CurrentStage": final_stage if status == "Hired" else status,
        "RecruitmentCost": agency_cost,
        "Gender": np.random.choice(["Female", "Male", "Non-Binary", "Prefer not to say"], p=[0.41, 0.52, 0.03, 0.04]),
        "Ethnicity": np.random.choice(["White", "Asian", "Hispanic", "Black", "Two or More"], p=[0.55, 0.22, 0.11, 0.09, 0.03])
    })

fact_applications = pd.DataFrame(apps_list)
fact_interview_logs = pd.DataFrame(interview_logs)

# Save to CSV
dim_recruiters.to_csv("Dim_Recruiters.csv", index=False)
dim_jobs.to_csv("Dim_Jobs.csv", index=False)
fact_applications.to_csv("Fact_Applications.csv", index=False)
fact_interview_logs.to_csv("Fact_Interview_Logs.csv", index=False)

print("Data Architecture files generated successfully: Dim_Recruiters, Dim_Jobs, Fact_Applications, Fact_Interview_Logs")

Data Architecture files generated successfully: Dim_Recruiters, Dim_Jobs, Fact_Applications, Fact_Interview_Logs
